In [1]:
import sys
print(sys.executable)

c:\Users\divya\Desktop\AI\LangChain_Lab\.venv\Scripts\python.exe


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from common.llm import get_llm

In [3]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

print("MODEL:", os.getenv("MODEL"))
print("GOOGLE KEY:", bool(os.getenv("GROQ_API_KEY")))

MODEL: openai/gpt-oss-20b
GOOGLE KEY: True


In [4]:
llm = get_llm()

response = llm.invoke("Say hi in one line")

response
# print(response.content)

AIMessage(content='Hi!', additional_kwargs={'reasoning_content': 'User says: "Say hi in one line". So presumably respond with a greeting in one line. Probably "Hi!" or "Hello!" The instruction: "Say hi in one line". So output: "Hi!". Probably that\'s it.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 76, 'total_tokens': 136, 'completion_time': 0.075577329, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.00424487, 'prompt_tokens_details': None, 'queue_time': 0.210685127, 'total_time': 0.079822199}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d6c0-a0c0-7b82-b0f0-ef932fd52e84-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 60, 'total_tokens': 136, 'output_token_details': {'reasoning': 49}})

In [5]:
response1 = llm.invoke("What is PostgreSQL?")


In [6]:
print(response1.content)

**PostgreSQL** (often called “Postgres”) is a powerful, open‑source **relational database management system (RDBMS)** that is widely used for both small projects and large, mission‑critical applications.

---

## 1.  Core Identity

| Feature | What it means |
|---------|---------------|
| **Open‑source** | Released under the PostgreSQL License (a permissive BSD‑style license). No licensing fees, and anyone can modify, distribute, or embed it in other products. |
| **ACID‑compliant** | Guarantees Atomicity, Consistency, Isolation, and Durability for transactions, ensuring reliable data handling. |
| **SQL‑standard compliant** | Implements most of the SQL:2011 standard, with many extensions. |
| **Extensible** | Users can add new data types, operators, index methods, procedural languages, and even write custom extensions in C or other languages. |
| **Multi‑model** | While primarily relational, it supports JSON/JSONB, XML, hstore, arrays, and even full‑text search out of the box. |
| **C

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [8]:
# ChatPromptTemplate (class)
# from_message() (class method)
# prompt = ... (object/instance)

# list
# │
# ├── tuple
# │   ├── "system"
# │   └── system prompt
# │
# └── tuple
#     ├── "human"
#     └── "{text}"

# system → system message
# human  → human/user message

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a rewriting assistant. Rewrite the user's text in a {tone} tone. "
     "Keep the original meaning exactly. Use at least {min_words} and at most {max_words} words. "
     "Output ONLY the rewritten text: no preamble, no explanation, no quotes."),
    ("human", "{text}"),
])

print(type(prompt))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [9]:
dir(prompt)

['InputType',
 'OutputType',
 '__abstractmethods__',
 '__add__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__parameters__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '

In [10]:
sample = "Hey, tomorrow's meeting is cancelled. Please tell everyone."

filled = prompt.invoke({"tone": "formal", "min_words": 50, "max_words": 100, "text": sample})

for m in filled.to_messages():
    print(f"[{m.type}] {m.content}\n")

[system] You are a rewriting assistant. Rewrite the user's text in a formal tone. Keep the original meaning exactly. Use at least 50 and at most 100 words. Output ONLY the rewritten text: no preamble, no explanation, no quotes.

[human] Hey, tomorrow's meeting is cancelled. Please tell everyone.



In [11]:
llm = get_llm(temperature=0.7)
chain = prompt | llm | StrOutputParser()

chain.invoke({"tone": "formal", "min_words": 50,  "max_words": 100, "text": sample})

'Dear colleagues, I regret to inform you that the meeting scheduled for tomorrow has been cancelled. Kindly inform all relevant parties of this change at your earliest convenience. Should there be any questions or need for rescheduling, please do not hesitate to contact me. Thank you for your prompt attention to this matter.'

In [12]:
def rewrite(text: str, tone: str = "formal", min_words: int = 20, max_words: int = 40, temperature: float = 0.7) -> str:
    chain = prompt | get_llm(temperature) | StrOutputParser()
    return chain.invoke({"text": text, "tone": tone, "min_words": min_words, "max_words": max_words})

In [13]:
tones = ["formal", "casual", "funny", "professional email"]

for tone in tones:
    print(f"--- {tone.upper()} ---")
    print(rewrite(sample, tone=tone, min_words = 20, max_words=40))
    print()

--- FORMAL ---
Dear all, please be advised that the meeting scheduled for tomorrow has been cancelled. Kindly inform all relevant parties of this change. Thank you.

--- CASUAL ---
Hey folks, just a heads‑up: the meeting tomorrow is off. Could you let everyone know and update the calendar? Thanks a bunch!

--- FUNNY ---
Attention, folks! The grand assembly scheduled for tomorrow has been officially cancelled. Spread the word faster than a cat video on the internet—everyone needs to know!

--- PROFESSIONAL EMAIL ---
Please be advised that the meeting scheduled for tomorrow has been cancelled. Kindly inform all participants of this change. Thank you.



In [14]:
for temp in [0, 0.9]:
    print(f"=== temperature = {temp} ===")

    for i in  range(3):
        print(f"{i+1}. {rewrite(sample, tone='funny', min_words = 20, max_words=30, temperature=temp)}")
    print()

=== temperature = 0 ===
1. Yo, the grand assembly scheduled for tomorrow has been officially scrapped. Kindly spread the word faster than a gossiping squirrel. Everyone, rejoice!
2. Yo, the grand assembly scheduled for tomorrow has been officially scrapped. Kindly spread the word faster than a gossiping squirrel. Everyone, rejoice!
3. Yo, the grand assembly scheduled for tomorrow has been officially scrapped. Kindly spread the word faster than a gossiping squirrel. Everyone, rejoice!

=== temperature = 0.9 ===
1. Good news, folks—tomorrow's meeting has been cancelled! So, grab your snacks, skip the chair, and let’s all enjoy a day of spontaneous productivity. Spread the word!
2. Attention, folks: tomorrow's meeting has been officially scrapped. Spread the word faster than a squirrel on espresso—everyone needs to know!
3. Listen up, team—tomorrow's meeting has been poofed away. Spread the word faster than a squirrel on a caffeine high! Right now.



In [15]:
chain = prompt | get_llm(0.7) | StrOutputParser()
inputs = {"tone": "professional email", "min_words": 100, "max_words": 200, "text": sample}

print("INVOKE:")
print(chain.invoke(inputs))

print("\nSTREAM:")
for chunk in chain.stream(inputs):
    print(chunk, end="", flush=True)

INVOKE:
Dear Team,

I hope this message finds you well. I am writing to inform you that the meeting scheduled for tomorrow has been cancelled. Please update your calendars accordingly and disregard any prior reminders or agendas that were circulated for that session.

If you had any specific items you intended to discuss during the meeting, please feel free to email me directly or bring them to our next scheduled gathering. I will be happy to ensure that any outstanding topics are addressed in a timely manner.

Thank you for your understanding and flexibility. Should you have any questions or require further clarification, do not hesitate to reach out.

Best regards,

[Your Name]

STREAM:
Subject: Cancellation of Tomorrow's Meeting

Dear Team,

I am writing to inform you that the meeting scheduled for tomorrow has been cancelled. Please update your calendars accordingly. If you had any agenda items or materials prepared, kindly resubmit them by [date]. Should you have any questions, fe

In [16]:
for n in [10, 25, 60]:
    out = rewrite(sample, tone="formal", min_words=10, max_words=n)
    print(f"max_words={n} -> actual={len(out.split())} words")
    print(out, "\n")

max_words=10 -> actual=10 words
Tomorrow's meeting has been cancelled; please inform all participants immediately. 

max_words=25 -> actual=10 words
Tomorrow's meeting has been cancelled; please inform all participants promptly. 

max_words=60 -> actual=16 words
Please be advised that the meeting scheduled for tomorrow has been cancelled. Kindly inform all participants. 



#### FULL FLOW

```Text
dict {tone, max_words, text}
        │
        ▼
   ChatPromptTemplate     →  system + human messages ban gaye
        │
        ▼
   Chat Model (llm)       →  AIMessage return hua (poora object, content + metadata)
        │
        ▼
   StrOutputParser        →  sirf .content nikal ke plain string di
        │
        ▼
   final string